### Import & Load Data

In [4]:
import pandas as pd
import numpy as np
import re
import time
import multiprocessing
from multiprocessing.pool import ThreadPool
import ast # <-- เติมบรรทัดนี้เข้ามาครับ

print("Loading dataset...")
df = pd.read_csv('../../data_resource/recipes.csv')
print(f"Dataset loaded: {len(df)} rows.")

Loading dataset...
Dataset loaded: 522517 rows.


### Preprocessing Function

In [5]:
def preprocess_data(df_chunk):
    for col in ['RecipeIngredientParts', 'RecipeInstructions']:
        df_chunk[col] = (df_chunk[col].fillna('')
                                      .astype(str)
                                      .str.replace(r'^c\((.*)\)$', r'\1', regex=True)
                                      .str.replace('character(0)', '', regex=False))

    df_chunk['Name'] = df_chunk['Name'].fillna('')

    combined = df_chunk['Name'] + " " + df_chunk['RecipeIngredientParts'] + " " + df_chunk['RecipeInstructions']

    df_chunk['clean_text'] = combined.str.lower().apply(lambda x: re.sub(r'[^\w\s]', '', x))

    return df_chunk

### Multiprocessing Benchmark

In [6]:
print("Starting preprocessing benchmark...")

c = multiprocessing.cpu_count()
print(f"Maximum cores available: {c}")

df_split = np.array_split(df, c)

print("Running with 1 core...")
start_1 = time.time()
with ThreadPool(1) as pool:
    result_1 = pool.map(preprocess_data, df_split)
time_1 = time.time() - start_1
print(f"Execution time (1 core): {time_1:.4f} s")

print(f"Running with {c} cores...")
start_n = time.time()
with ThreadPool(c) as pool:
    result_n = pool.map(preprocess_data, df_split)
time_n = time.time() - start_n
print(f"Execution time ({c} cores): {time_n:.4f} s")

speedup = time_1 / time_n
print(f"Speedup: {speedup:.2f}x")

print("\n--- Starting Image Imputation ---")
processed_df = pd.concat(result_n)

def extract_first_image(image_str):
    if pd.isna(image_str) or image_str in ['character(0)', 'c()', '[]']:
        return np.nan
    try:
        if isinstance(image_str, str) and image_str.startswith('['):
            images = ast.literal_eval(image_str)
            if isinstance(images, list) and len(images) > 0:
                return images[0]
        elif isinstance(image_str, str) and 'http' in image_str:
            urls = [url for url in image_str.split('"') if url.startswith('http')]
            return urls[0] if urls else np.nan
    except:
        pass
    return np.nan

processed_df['Image_URL'] = processed_df['Images'].apply(extract_first_image)
print(f"Initial missing images: {processed_df['Image_URL'].isna().sum()}")

valid_images = processed_df.dropna(subset=['Image_URL'])
name_to_img = valid_images.groupby('Name')['Image_URL'].first().to_dict()

processed_df['Image_URL'] = processed_df['Image_URL'].fillna(processed_df['Name'].map(name_to_img))

if 'RecipeCategory' in processed_df.columns:
    cat_to_img = valid_images.groupby('RecipeCategory')['Image_URL'].first().to_dict()
    processed_df['Image_URL'] = processed_df['Image_URL'].fillna(processed_df['RecipeCategory'].map(cat_to_img))

DEFAULT_IMG = "https://images.unsplash.com/photo-1495195134817-a169d25fb25b?w=800&q=80"
processed_df['Image_URL'] = processed_df['Image_URL'].fillna(DEFAULT_IMG)

print(f"Final missing images: {processed_df['Image_URL'].isna().sum()} (Should be 0!)")
print("--------------------------------------\n")

print("Saving preprocessed data...")

cols_to_keep = [
    'RecipeId',
    'Name',
    'RecipeCategory',
    'RecipeIngredientParts',
    'RecipeInstructions',
    'Image_URL',
    'clean_text'
]

available_cols = [col for col in cols_to_keep if col in processed_df.columns]

processed_df[available_cols].to_csv('../../data_resource/preprocessed_recipes.csv', index=False)
print("Process completed.")

Starting preprocessing benchmark...
Maximum cores available: 12


C:\Users\Nitro V15\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Running with 1 core...
Execution time (1 core): 36.4212 s
Running with 12 cores...
Execution time (12 cores): 45.9119 s
Speedup: 0.79x

--- Starting Image Imputation ---
Initial missing images: 356621
Final missing images: 0 (Should be 0!)
--------------------------------------

Saving preprocessed data...
Process completed.
